# Problema del Peluquero Dormido — Análisis Formal y Verificación Experimental

**Universidad del Valle — Departamento de Ingeniería de Sistemas**  
**Sistemas Operativos — Taller Final**  

---

## Frente 1 — Reproducción Determinista del Fallo (*Lost Wakeup*)

### 1. Objetivo
El objetivo de esta sección es reproducir e ilustrar de manera completamente determinista el fallo de concurrencia conocido como **Lost Wakeup** (Despertar Perdido) en la versión incorrecta del problema del peluquero dormido, donde no se emplea una sincronización atómica para la secuencia de verificación de estado y bloqueo (dormir).

## 2. Fundamento Teórico

El fenómeno del **Lost Wakeup** ocurre cuando un hilo lector/receptor se prepara para bloquearse esperando una señal, pero justo antes de efectuar la llamada de bloqueo (p. ej., `wait()`), es interrumpido por el planificador (scheduler). Otro hilo (el emisor) se ejecuta, altera la condición de espera y envía la señal de despertar (`signal` o `set`). Como el primer hilo aún no está técnicamente en estado de espera, la señal se pierde. Cuando el primer hilo reanuda su ejecución, procede a bloquearse y queda suspendido indefinidamente porque la señal ya fue emitida y no volverá a ocurrir.

En el problema del peluquero dormido:
1. El barbero comprueba `waiting == 0` (no hay clientes).
2. Justo antes de irse a dormir (ejecutar `wait()`), llega un cliente.
3. El cliente ve que el barbero está "despierto" (pues el estado aún no es "durmiendo"), por lo que no le envía la señal de despertar (`set()`), limitándose a sentarse en la silla de espera.
4. El barbero reanuda su ejecución, asume que no hay clientes porque ya hizo la evaluación previa, y se duerme (`wait()`).
5. El barbero queda durmiendo indefinidamente (Lost Wakeup) y el cliente esperando en la silla para siempre.

In [5]:
# ===========================================================================
# FRENTE 1 — Implementación INCORRECTA del Problema del Peluquero Dormido
# Propósito: Demostrar el Lost Wakeup de forma determinista
# ===========================================================================

import threading
import time
import random

# Parámetros de la barbería
NUM_CHAIRS = 3          # Número de sillas de espera
INJECT_DELAY = 0.15     # Retardo inyectado para simular la interrupción del scheduler

# Variables de estado compartidas sin protección adecuada de exclusión mutua
waiting = 0             # Clientes esperando
barber_sleeping = False # Estado del barbero
barber_event = threading.Event()  # Evento para despertar al barbero

# Registro de eventos
start_time = time.perf_counter()
event_log = []
log_lock = threading.Lock()  # Exclusión mutua únicamente para el orden del log impreso

def log(actor, message):
    """Registra un evento con marca de tiempo precisa."""
    with log_lock:
        ts = time.perf_counter() - start_time
        entry = f"[T={ts:.4f}s] [{actor}] {message}"
        event_log.append(entry)
        print(entry)

def barber_incorrect():
    global waiting, barber_sleeping
    
    log("BARBERO", "Inicio de turno. Verificando si hay clientes...")
    
    # Verificación del estado de la sala
    # Entre esta lectura de 'waiting' y el bloqueo físico en 'wait()'
    # ocurre la interrupción del scheduler (ventana de vulnerabilidad).
    if waiting == 0:
        log("BARBERO", "No hay clientes. Me preparo para dormir...")
        
        # INYECCIÓN DEL RETARDO:
        # Forzamos al scheduler a dar paso a otro hilo (el cliente) antes de bloquearnos.
        time.sleep(INJECT_DELAY)
        
        # Transición al estado durmiendo y bloqueo
        barber_sleeping = True
        log("BARBERO", "Entrando en estado durmiendo. Ejecutando wait()...")
        
        # Usamos un timeout de 2.0 segundos para el experimento
        woke_up = barber_event.wait(timeout=2.0)
        
        if woke_up:
            barber_sleeping = False
            log("BARBERO", "Desperté. Atendiendo al cliente.")
        else:
            log("BARBERO", " FALLO: Nadie me despertó. Lost Wakeup confirmado.")
    else:
        log("BARBERO", f"Hay {waiting} cliente(s) esperando. Atendiendo...")

def client_incorrect(client_id, arrival_delay):
    global waiting, barber_sleeping
    
    # Simula el viaje del cliente a la barbería
    time.sleep(arrival_delay)
    log(f"CLIENTE-{client_id}", "Llegando a la barbería...")
    
    # Condicion para evaluar sillas libres
    if waiting < NUM_CHAIRS:
        waiting += 1
        log(f"CLIENTE-{client_id}", f"Me senté en una silla. Esperando = {waiting}")
        
        # El cliente comprueba si el barbero duerme
        # Como el barbero está en su ventana de retardo, barber_sleeping es aún False.
        # El cliente asume erróneamente que el barbero está despierto y NO envía la señal.
        if barber_sleeping:
            log(f"CLIENTE-{client_id}", "El barbero está durmiendo. Enviando señal de despertar (set)... ")
            barber_event.set()
        else:
            log(f"CLIENTE-{client_id}", "Barbero despierto. NO envío señal. [AQUI OCURRE EL FALLO]")
    else:
        log(f"CLIENTE-{client_id}", "Sala llena. Me retiro.")

### 3. Ejecución Experimental del Fallo

In [6]:
print("=== EXPERIMENTO: REPRODUCCIÓN DEL LOST WAKEUP ===\n")

# Limpieza e inicialización
event_log.clear()
barber_event.clear()
waiting = 0
barber_sleeping = False

# Hilos: El barbero inicia de inmediato (T=0). El cliente llega en T=0.05s, 
# cayendo exactamente en la ventana de vulnerabilidad del barbero (0s a 0.15s).
t_barber = threading.Thread(target=barber_incorrect)
t_client = threading.Thread(target=client_incorrect, args=(1, 0.05))

t_start = time.perf_counter()
t_barber.start()
t_client.start()

t_barber.join()
t_client.join()
t_duration = time.perf_counter() - t_start

print(f"\nDuración de la simulación: {t_duration:.4f} segundos")
if t_duration >= 2.0:
    print("\nRESULTADO: [Fallo Confirmado] El barbero quedó bloqueado permanentemente (se superó el timeout).")
else:
    print("\nRESULTADO: El barbero se despertó a tiempo.")

=== EXPERIMENTO: REPRODUCCIÓN DEL LOST WAKEUP ===

[T=0.0064s] [BARBERO] Inicio de turno. Verificando si hay clientes...
[T=0.0064s] [BARBERO] No hay clientes. Me preparo para dormir...
[T=0.0616s] [CLIENTE-1] Llegando a la barbería...
[T=0.0617s] [CLIENTE-1] Me senté en una silla. Esperando = 1
[T=0.0617s] [CLIENTE-1] Barbero despierto. NO envío señal. [AQUI OCURRE EL FALLO]
[T=0.1615s] [BARBERO] Entrando en estado durmiendo. Ejecutando wait()...
[T=2.1636s] [BARBERO]  FALLO: Nadie me despertó. Lost Wakeup confirmado.

Duración de la simulación: 2.1583 segundos

RESULTADO: [Fallo Confirmado] El barbero quedó bloqueado permanentemente (se superó el timeout).


---

## Frente 2 — Modelo e Invariantes del Sistema

### 1. Modelo Formal del Sistema

Para analizar formalmente la concurrencia en la barbería, definimos los componentes del sistema:

#### 1.1 Actores y sus Estados
- **Barbero:** DURMIENDO, VERIFICANDO, TRABAJANDO.
- **Clientes:** VIAJANDO, LLEGANDO, ESPERANDO, RECIBIENDO_CORTE, ATENDIDO, RECHAZADO.

#### 1.2 Recursos Compartidos y Variables
- **Sala de Espera:** Capacidad limitada de $N$ sillas (recurso crítico).
- `sillas_espera`: Contador de clientes esperando en la sala.
- `barbero_durmiendo`: Bandera lógica del estado del barbero.

--- 

### 2. Invariantes del Sistema (Propiedades de Seguridad)

- **Invariante 1 (Capacidad de la Sala):**
  $$0 \le sillas\_espera \le CAPACIDAD\_MAX$$
- **Invariante 2 (Ausencia de Negligencia del Barbero):**
  $$\text{Si } barbero\_durmiendo = True \implies sillas\_espera = 0$$
- **Invariante 3 (Ausencia de Espera Inútil):**
  $$\text{Si } sillas\_espera > 0 \implies barbero\_durmiendo = False$$

### 3. Implementación Simplificada de Verificación de Invariantes

In [13]:
# ===========================================================================
# FRENTE 2 — FUNCIÓN DE VERIFICACIÓN DE INVARIANTES
# ===========================================================================

def verificar_invariantes(sillas_espera, barbero_durmiendo, capacidad_max=3):
    """Valida las aserciones de seguridad del sistema."""
    # Invariante 1: Capacidad de la sala
    assert 0 <= sillas_espera <= capacidad_max, \
        f"Violación: sillas_espera ({sillas_espera}) fuera de rango [0, {capacidad_max}]."
    
    # Invariante 2: Si el barbero duerme, la sala de espera debe estar vacía
    if barbero_durmiendo:
        assert sillas_espera == 0, \
            f"Violación: El barbero duerme pero hay {sillas_espera} clientes esperando."
            
    # Invariante 3: Si hay clientes esperando, el barbero no puede estar dormido
    if sillas_espera > 0:
        assert not barbero_durmiendo, \
            "Violación: Hay clientes esperando pero el barbero está dormido."

# ===========================================================================
# PRUEBA DE FUNCIONAMIENTO DE LOS INVARIANTES
# ===========================================================================
print("=== PRUEBA DE INVARIANTES (FRENTE 2) ===\n")

# Caso 1: Estado válido
print("1. Evaluando estado válido (1 cliente esperando, barbero despierto)... ")
verificar_invariantes(sillas_espera=1, barbero_durmiendo=False)
print("   Resultado: [OK] Estado seguro.")

# Caso 2: Estado inválido (fuerza un error)
print("\n2. Evaluando estado inválido (2 clientes esperando, barbero dormido)... ")
try:
    verificar_invariantes(sillas_espera=2, barbero_durmiendo=True)
    print("   Resultado: [ERROR] Se permitió un estado inconsistente.")
except AssertionError as e:
    print(f"   Resultado: [VIOLACIÓN DETECTADA CORRECTAMENTE] {e}")

=== PRUEBA DE INVARIANTES (FRENTE 2) ===

1. Evaluando estado válido (1 cliente esperando, barbero despierto)... 
   Resultado: [OK] Estado seguro.

2. Evaluando estado inválido (2 clientes esperando, barbero dormido)... 
   Resultado: [VIOLACIÓN DETECTADA CORRECTAMENTE] Violación: El barbero duerme pero hay 2 clientes esperando.


---

## Frente 3 — Implementación correcta

### 1. Justificación del Mecanismo de Sincronización

Utilizaremos una única variable de condición `threading.Condition()`. Esta primitiva envuelve un lock interno que garantiza exclusión mutua para leer y escribir sobre las variables globales, además de permitir la sincronización mediante `wait()` y `notify_all()` para evitar el Lost Wakeup.

##  Recursos Compartidos y Mecanismos de Sincronización

En este bloque se inicializan los recursos compartidos que utilizarán todos los hilos durante la simulación. Se define el **Lock**, encargado de garantizar la exclusión mutua sobre el estado compartido, y la **Variable de Condición (`Condition`)**, utilizada para coordinar el sueño y el despertar del barbero sin recurrir a espera activa (*Busy Waiting*).

También se inicializan las variables que representan el estado de la barbería, como el número de clientes esperando, el estado del barbero, la apertura de la barbería y los contadores de auditoría que permitirán verificar posteriormente el comportamiento de la simulación.

In [14]:
# ===========================================================================
# FRENTE 3 — IMPLEMENTACIÓN CORRECTA DEL PROBLEMA DEL PELUQUERO DORMIDO
# Chunk 1: Recursos Compartidos y Mecanismos de Sincronización
# ===========================================================================

import threading
import time
import random

# -------------------------------------------------------------
# Parámetros de la simulación
# -------------------------------------------------------------

NUM_CHAIRS = 3          # Número de sillas en la sala de espera
NUM_CLIENTS = 5         # Cantidad de clientes que llegarán

# -------------------------------------------------------------
# Lock principal
#
# Garantiza exclusión mutua sobre el estado compartido.
# Ningún hilo puede modificar simultáneamente la barbería.
# -------------------------------------------------------------

lock = threading.Lock()

# -------------------------------------------------------------
# Variable de Condición
#
# Permite bloquear y despertar hilos sin realizar espera activa
# (Busy Waiting).
#
# OSTEP Cap.30 recomienda utilizar siempre:
#
#       Lock + Condition + while
#
# -------------------------------------------------------------

condition = threading.Condition(lock)

# -------------------------------------------------------------
# Estado compartido de la barbería
# -------------------------------------------------------------

waiting = 0                 # Clientes esperando
barber_sleeping = False     # Estado del barbero
shop_open = True            # Permite finalizar la simulación

# -------------------------------------------------------------
# Variables de auditoría
# -------------------------------------------------------------

clients_served = 0
clients_rejected = 0

# -------------------------------------------------------------
# Registro de eventos
# -------------------------------------------------------------

start_time = time.perf_counter()

log_lock = threading.Lock()

def log(actor, message):
    """
    Imprime eventos con marca de tiempo.
    Facilita la explicación durante la sustentación.
    """

    with log_lock:

        ts = time.perf_counter() - start_time

        print(f"[T={ts:.4f}s] [{actor}] {message}")

## Implementación del Hilo del Barbero

Este bloque implementa el comportamiento del hilo que representa al barbero. El barbero permanece ejecutándose mientras la barbería se encuentre abierta y utiliza una **Variable de Condición** para suspender su ejecución cuando no existen clientes en espera.

Siguiendo la recomendación de **OSTEP (Capítulo 30)**, la espera se realiza dentro de un ciclo `while`, garantizando que la condición sea verificada nuevamente después de cada despertar. De esta manera, la implementación evita despertares espurios y elimina la posibilidad del problema conocido como **Lost Wakeup**.

Finalmente, el corte de cabello se realiza fuera de la sección crítica para permitir que otros clientes puedan llegar a la barbería de forma concurrente sin bloquear el acceso a los recursos compartidos.

In [15]:
# ===========================================================================
# Chunk 2: Hilo del Barbero
# ===========================================================================

def barber_correct():
    """
    Implementación correcta del hilo del barbero.

    Se utiliza una Variable de Condición (Condition) protegida por un Lock.

    La combinación:

            Lock + Condition + while

    evita el problema de Lost Wakeup descrito en OSTEP
    (Capítulo 30).
    """

    global waiting
    global barber_sleeping
    global clients_served
    global shop_open

    log("BARBERO", "Inicio de turno.")

    while True:

        # ---------------------------------------------------------
        # Sección crítica
        #
        # Solo un hilo puede acceder al estado compartido
        # (waiting, barber_sleeping, etc.) al mismo tiempo.
        # ---------------------------------------------------------

        with condition:

            # -----------------------------------------------------
            # CONDICIÓN DE ESPERA
            #
            # Mientras no existan clientes y la barbería permanezca
            # abierta, el barbero libera el Lock y entra en espera.
            #
            # Se utiliza WHILE (y no IF) por dos razones:
            #
            # 1. Evitar despertares espurios.
            # 2. Revalidar la condición cuando recupera el Lock.
            #
            # (OSTEP - Capítulo 30)
            # -----------------------------------------------------

            while waiting == 0 and shop_open:

                barber_sleeping = True

                log(
                    "BARBERO",
                    "No hay clientes. Esperando..."
                )

                condition.wait()

            # -----------------------------------------------------
            # CONDICIÓN DE PARADA
            #
            # El hilo principal marcará shop_open=False cuando
            # todos los clientes hayan finalizado.
            #
            # Si ya no existen clientes pendientes,
            # el barbero termina ordenadamente.
            # -----------------------------------------------------

            if not shop_open and waiting == 0:

                log(
                    "BARBERO",
                    "Fin del turno."
                )

                break

            # -----------------------------------------------------
            # Existe al menos un cliente esperando.
            #
            # El barbero toma un cliente de la sala.
            # -----------------------------------------------------

            barber_sleeping = False

            waiting -= 1

            clients_served += 1

            log(
                "BARBERO",
                f"Comienza a atender. Clientes esperando = {waiting}"
            )

        # ---------------------------------------------------------
        # CORTE DE CABELLO
        #
        # El corte ocurre FUERA de la sección crítica.
        #
        # De esta forma otros clientes pueden seguir llegando
        # mientras el barbero trabaja.
        # ---------------------------------------------------------

        time.sleep(random.uniform(0.10,0.30))

        log(
            "BARBERO",
            "Corte finalizado."
        )

In [16]:
# ===========================================================================
# Chunk 3: Hilo del Cliente
# ===========================================================================

def client_correct(client_id, arrival_delay):
    """
    Implementación correcta del hilo cliente.

    Cada cliente llega a la barbería de manera concurrente,
    intenta ocupar una silla de espera y, si el barbero se
    encuentra dormido, lo despierta mediante notify().

    Todas las operaciones sobre el estado compartido se
    realizan dentro de la sección crítica protegida por
    el Lock asociado a la Variable de Condición.
    """

    global waiting
    global barber_sleeping
    global clients_rejected

    # ---------------------------------------------------------
    # Simulación del tiempo de llegada del cliente
    # ---------------------------------------------------------

    time.sleep(arrival_delay)

    log(
        f"CLIENTE-{client_id}",
        "Llegando a la barbería."
    )

    # ---------------------------------------------------------
    # Sección crítica
    # ---------------------------------------------------------

    with condition:

        # -----------------------------------------------------
        # Verificación de espacio disponible.
        #
        # Si la sala está llena, el cliente abandona
        # la barbería inmediatamente.
        # -----------------------------------------------------

        if waiting >= NUM_CHAIRS:

            clients_rejected += 1

            log(
                f"CLIENTE-{client_id}",
                "Sala llena. Me retiro."
            )

            return

        # -----------------------------------------------------
        # El cliente ocupa una silla de espera.
        # -----------------------------------------------------

        waiting += 1

        log(
            f"CLIENTE-{client_id}",
            f"Esperando. Clientes en sala = {waiting}"
        )

        # -----------------------------------------------------
        # Si el barbero estaba dormido,
        # el cliente lo despierta.
        #
        # notify() solamente informa que la condición
        # cambió. El barbero deberá volver a verificar
        # waiting dentro del while.
        # -----------------------------------------------------

        if barber_sleeping:

            log(
                f"CLIENTE-{client_id}",
                "Despertando al barbero."
            )

            condition.notify()

    # ---------------------------------------------------------
    # El cliente termina su ejecución.
    #
    # La atención será realizada por el hilo del barbero.
    # -----------------------------------------------------

    log(
        f"CLIENTE-{client_id}",
        "Esperando su turno."
    )

## Creación de los Hilos

En este bloque se crean los hilos que participarán en la simulación. Se instancia un único hilo para representar al barbero y varios hilos para representar a los clientes, cada uno con un tiempo de llegada aleatorio para simular un escenario de concurrencia.

En esta etapa únicamente se crean los hilos; la ejecución de la simulación se realizará en el siguiente bloque.

In [17]:
# ===========================================================================
# Chunk 4: Creación de los Hilos
# ===========================================================================

print("=== SIMULACIÓN: IMPLEMENTACIÓN CORRECTA DEL PELUQUERO DORMIDO ===\n")

# -------------------------------------------------------------
# Reinicio del estado compartido
# -------------------------------------------------------------

waiting = 0
barber_sleeping = False
shop_open = True

clients_served = 0
clients_rejected = 0

start_time = time.perf_counter()

# -------------------------------------------------------------
# Declaración del hilo del barbero
#
# Existe un único barbero durante toda la simulación.
# Permanecerá ejecutándose hasta que la barbería cierre.
# -------------------------------------------------------------

barber = threading.Thread(
    target=barber_correct,
    name="BARBERO"
)

# -------------------------------------------------------------
# Declaración de los hilos clientes
#
# Cada cliente es un hilo independiente que llega
# después de un pequeño retardo aleatorio.
# -------------------------------------------------------------

clients = []

for i in range(NUM_CLIENTS):

    arrival = random.uniform(0.0,0.5)

    client = threading.Thread(
        target=client_correct,
        args=(i + 1, arrival),
        name=f"CLIENTE-{i+1}"
    )

    clients.append(client)

=== SIMULACIÓN: IMPLEMENTACIÓN CORRECTA DEL PELUQUERO DORMIDO ===



## Ejecución de la Simulación

En este bloque se inicia la ejecución concurrente de todos los hilos. Primero se inicia el hilo del barbero y posteriormente los hilos clientes.

El hilo principal utiliza `join()` para esperar la finalización de todos los clientes. Una vez atendidos o rechazados, se cambia la condición de parada (`shop_open = False`) y se despierta al barbero mediante `notify()` para que finalice su ejecución de forma ordenada.

Finalmente se imprime un resumen con el número de clientes atendidos y rechazados.

In [18]:
# ===========================================================================
# Chunk 5: Ejecución de la Simulación
# ===========================================================================

# -------------------------------------------------------------
# Inicio del hilo del barbero
# -------------------------------------------------------------

barber.start()

# -------------------------------------------------------------
# Inicio de los clientes
# -------------------------------------------------------------

for client in clients:
    client.start()

# -------------------------------------------------------------
# Esperar que todos los clientes finalicen
#
# join() garantiza que el hilo principal espere
# hasta que cada cliente termine su ejecución.
# -------------------------------------------------------------

for client in clients:
    client.join()

# -------------------------------------------------------------
# Condición de parada
#
# Una vez finalizados todos los clientes,
# la barbería deja de aceptar nuevas llegadas.
#
# Se despierta al barbero para que salga del wait()
# y finalice su ejecución.
# -------------------------------------------------------------

with condition:

    shop_open = False

    condition.notify()

# -------------------------------------------------------------
# Esperar la finalización del barbero
# -------------------------------------------------------------

barber.join()

# -------------------------------------------------------------
# Resumen de la simulación
# -------------------------------------------------------------

print("\n==========================================")
print("RESULTADOS DE LA SIMULACIÓN")
print("==========================================")

print(f"Clientes atendidos : {clients_served}")
print(f"Clientes rechazados: {clients_rejected}")
print(f"Clientes restantes : {waiting}")

print("\nImplementación finalizada correctamente.")

[T=0.0049s] [BARBERO] Inicio de turno.
[T=0.0053s] [BARBERO] No hay clientes. Esperando...
[T=0.0267s] [CLIENTE-3] Llegando a la barbería.
[T=0.0267s] [CLIENTE-3] Esperando. Clientes en sala = 1
[T=0.0267s] [CLIENTE-3] Despertando al barbero.
[T=0.0268s] [CLIENTE-3] Esperando su turno.
[T=0.0268s] [BARBERO] Comienza a atender. Clientes esperando = 0
[T=0.0620s] [CLIENTE-2] Llegando a la barbería.
[T=0.0621s] [CLIENTE-2] Esperando. Clientes en sala = 1
[T=0.0621s] [CLIENTE-2] Esperando su turno.
[T=0.1201s] [CLIENTE-1] Llegando a la barbería.
[T=0.1201s] [CLIENTE-1] Esperando. Clientes en sala = 2
[T=0.1201s] [CLIENTE-1] Esperando su turno.
[T=0.2253s] [BARBERO] Corte finalizado.
[T=0.2255s] [BARBERO] Comienza a atender. Clientes esperando = 1
[T=0.2866s] [CLIENTE-4] Llegando a la barbería.
[T=0.2868s] [CLIENTE-4] Esperando. Clientes en sala = 2
[T=0.2868s] [CLIENTE-4] Esperando su turno.
[T=0.3440s] [BARBERO] Corte finalizado.
[T=0.3441s] [BARBERO] Comienza a atender. Clientes esperand

 # Frente 4 — Pruebas de estrés y verificación: Harness con aserciones y casos adversos que intenten romper la solución.

# Frente 4 — Pruebas de Estrés y Verificación

##  Objetivo del Harness

En este frente se verifica que la implementación correcta preserve los invariantes definidos en el Frente 2 bajo distintos escenarios de concurrencia.

El propósito del *Harness* es ejecutar automáticamente varios casos de prueba que intentan llevar el sistema a situaciones límite, comprobando mediante aserciones que la solución mantiene las propiedades de **Seguridad (Safety)**, **Vivacidad (Liveness)** y **Equidad (Fairness)**.

Cada escenario ejecutará la simulación con diferentes patrones de llegada de clientes y verificará que:

- Nunca existan más clientes esperando que sillas disponibles.
- El barbero nunca permanezca dormido existiendo clientes en espera.
- Todos los clientes sean correctamente atendidos o rechazados.
- La simulación finalice sin producir interbloqueos (*Deadlocks*).

In [19]:
# ===========================================================================
# CHUNK 2 — FUNCIÓN GENERAL DE SIMULACIÓN
# ===========================================================================

def ejecutar_simulacion(num_clientes, llegada_min, llegada_max):

    global waiting
    global barber_sleeping
    global shop_open
    global clients_served
    global clients_rejected

    print("\n" + "="*60)

    print(f"Simulación con {num_clientes} clientes")

    print("="*60)

    # Reiniciar estado compartido

    waiting = 0
    barber_sleeping = False
    shop_open = True

    clients_served = 0
    clients_rejected = 0

    # Crear hilo del barbero

    barber = threading.Thread(target=barber_correct)

    barber.start()

    # Crear clientes

    clientes = []

    for i in range(num_clientes):

        llegada = random.uniform(
            llegada_min,
            llegada_max
        )

        hilo = threading.Thread(
            target=client_correct,
            args=(i+1, llegada)
        )

        clientes.append(hilo)

        hilo.start()

    # Esperar clientes

    for hilo in clientes:

        hilo.join()

    # Cerrar barbería

    with condition:

        shop_open = False

        condition.notify()

    barber.join()

    print("\nResumen")

    print(f"Atendidos : {clients_served}")

    print(f"Rechazados: {clients_rejected}")

    print(f"Esperando : {waiting}")

    verificar_invariantes(
        waiting,
        barber_sleeping
    )

    print("✔ Simulación finalizada correctamente.")

### Caso base

In [20]:
print("CASO 1")

ejecutar_simulacion(

    num_clientes=5,

    llegada_min=0.2,

    llegada_max=0.5

)

CASO 1

Simulación con 5 clientes
[T=4515.4856s] [BARBERO] Inicio de turno.
[T=4515.4856s] [BARBERO] No hay clientes. Esperando...
[T=4515.6915s] [CLIENTE-2] Llegando a la barbería.
[T=4515.6919s] [CLIENTE-2] Esperando. Clientes en sala = 1
[T=4515.6920s] [CLIENTE-2] Despertando al barbero.
[T=4515.6920s] [CLIENTE-2] Esperando su turno.
[T=4515.6920s] [BARBERO] Comienza a atender. Clientes esperando = 0
[T=4515.7227s] [CLIENTE-5] Llegando a la barbería.
[T=4515.7228s] [CLIENTE-5] Esperando. Clientes en sala = 1
[T=4515.7228s] [CLIENTE-5] Esperando su turno.
[T=4515.8783s] [BARBERO] Corte finalizado.
[T=4515.8784s] [BARBERO] Comienza a atender. Clientes esperando = 0
[T=4515.8912s] [CLIENTE-3] Llegando a la barbería.
[T=4515.8913s] [CLIENTE-3] Esperando. Clientes en sala = 1
[T=4515.8913s] [CLIENTE-3] Esperando su turno.
[T=4515.9043s] [CLIENTE-1] Llegando a la barbería.
[T=4515.9053s] [CLIENTE-1] Esperando. Clientes en sala = 2
[T=4515.9053s] [CLIENTE-1] Esperando su turno.
[T=4515.958

In [21]:
print("CASO 2")

ejecutar_simulacion(

    num_clientes=5,

    llegada_min=0,

    llegada_max=0

)

CASO 2

Simulación con 5 clientes
[T=4516.8727s] [BARBERO] Inicio de turno.
[T=4516.8728s] [BARBERO] No hay clientes. Esperando...
[T=4516.8730s] [CLIENTE-1] Llegando a la barbería.
[T=4516.8730s] [CLIENTE-1] Esperando. Clientes en sala = 1
[T=4516.8730s] [CLIENTE-1] Despertando al barbero.
[T=4516.8730s] [CLIENTE-1] Esperando su turno.
[T=4516.8731s] [BARBERO] Comienza a atender. Clientes esperando = 0
[T=4516.8732s] [CLIENTE-2] Llegando a la barbería.
[T=4516.8732s] [CLIENTE-2] Esperando. Clientes en sala = 1
[T=4516.8732s] [CLIENTE-2] Esperando su turno.
[T=4516.8736s] [CLIENTE-4] Llegando a la barbería.
[T=4516.8736s] [CLIENTE-4] Esperando. Clientes en sala = 2
[T=4516.8736s] [CLIENTE-4] Esperando su turno.
[T=4516.8738s] [CLIENTE-5] Llegando a la barbería.
[T=4516.8738s] [CLIENTE-5] Esperando. Clientes en sala = 3
[T=4516.8738s] [CLIENTE-5] Esperando su turno.
[T=4516.8739s] [CLIENTE-3] Llegando a la barbería.
[T=4516.8739s] [CLIENTE-3] Sala llena. Me retiro.
[T=4517.1181s] [BARBE

In [22]:
print("CASO 3")

NUM_CHAIRS = 2

ejecutar_simulacion(

    num_clientes=10,

    llegada_min=0,

    llegada_max=0.2

)

CASO 3

Simulación con 10 clientes
[T=4517.8007s] [BARBERO] Inicio de turno.
[T=4517.8008s] [BARBERO] No hay clientes. Esperando...
[T=4517.8062s] [CLIENTE-1] Llegando a la barbería.
[T=4517.8062s] [CLIENTE-1] Esperando. Clientes en sala = 1
[T=4517.8063s] [CLIENTE-1] Despertando al barbero.
[T=4517.8063s] [CLIENTE-1] Esperando su turno.
[T=4517.8063s] [BARBERO] Comienza a atender. Clientes esperando = 0
[T=4517.8136s] [CLIENTE-3] Llegando a la barbería.
[T=4517.8137s] [CLIENTE-3] Esperando. Clientes en sala = 1
[T=4517.8137s] [CLIENTE-3] Esperando su turno.
[T=4517.8137s] [CLIENTE-2] Llegando a la barbería.
[T=4517.8137s] [CLIENTE-2] Esperando. Clientes en sala = 2
[T=4517.8137s] [CLIENTE-2] Esperando su turno.
[T=4517.8429s] [CLIENTE-9] Llegando a la barbería.
[T=4517.8429s] [CLIENTE-9] Sala llena. Me retiro.
[T=4517.9049s] [CLIENTE-4] Llegando a la barbería.
[T=4517.9052s] [CLIENTE-4] Sala llena. Me retiro.
[T=4517.9087s] [CLIENTE-7] Llegando a la barbería.
[T=4517.9087s] [CLIENTE-7

### Estrés

In [23]:
print("CASO 4")

NUM_CHAIRS = 5

ejecutar_simulacion(

    num_clientes=100,

    llegada_min=0,

    llegada_max=0.02

)

CASO 4

Simulación con 100 clientes
[T=4518.5404s] [BARBERO] Inicio de turno.
[T=4518.5404s] [BARBERO] No hay clientes. Esperando...
[T=4518.5434s] [CLIENTE-6] Llegando a la barbería.
[T=4518.5434s] [CLIENTE-6] Esperando. Clientes en sala = 1
[T=4518.5434s] [CLIENTE-6] Despertando al barbero.
[T=4518.5435s] [CLIENTE-6] Esperando su turno.
[T=4518.5436s] [BARBERO] Comienza a atender. Clientes esperando = 0
[T=4518.5453s] [CLIENTE-10] Llegando a la barbería.
[T=4518.5454s] [CLIENTE-10] Esperando. Clientes en sala = 1
[T=4518.5454s] [CLIENTE-10] Esperando su turno.
[T=4518.5455s] [CLIENTE-31] Llegando a la barbería.
[T=4518.5455s] [CLIENTE-31] Esperando. Clientes en sala = 2
[T=4518.5455s] [CLIENTE-31] Esperando su turno.
[T=4518.5455s] [CLIENTE-28] Llegando a la barbería.
[T=4518.5456s] [CLIENTE-28] Esperando. Clientes en sala = 3
[T=4518.5456s] [CLIENTE-28] Esperando su turno.
[T=4518.5465s] [CLIENTE-14] Llegando a la barbería.
[T=4518.5465s] [CLIENTE-14] Esperando. Clientes en sala = 4

# Frente 5 — Medición y Análisis del Desempeño


## Objetivo de la Medición

Hasta este punto se ha demostrado que la solución elimina el problema del **Lost Wakeup** y preserva los invariantes del sistema bajo distintos escenarios de concurrencia.

En este frente se realizan mediciones cuantitativas para evaluar el desempeño de la implementación. Se analizarán tres métricas principales:

- **Tiempo de espera de los clientes:** tiempo transcurrido desde que un cliente llega a la barbería hasta que comienza a ser atendido.
- **Throughput:** cantidad de clientes atendidos por segundo durante la simulación.
- **Contención:** número de clientes que debieron esperar debido a que el recurso compartido (el barbero) ya estaba ocupado.

Estas métricas permiten respaldar experimentalmente que la solución no solo es correcta desde el punto de vista de la sincronización, sino también eficiente bajo diferentes niveles de carga.

In [ ]:
# ===========================================================================
# CHUNK 2 — Instrumentación de Métricas
# ===========================================================================

# Tiempo de llegada de cada cliente
arrival_times = {}

# Momento en que el barbero comienza a atenderlo
service_start = {}

# Tiempo individual de espera
waiting_times = []

# Número de clientes que tuvieron que esperar
clients_waited = 0